# Phishing Cross-Dataset Feature Robustness Study — Training & Evaluation  
**Project Goal**: Investigate model performance and feature robustness across diverse phishing datasets.  
  **Models:**
     1. **Logistic Regression:** Baseline linear classifier  
     2. **Random Forest:** Tree-based ensemble (robust to mixed features)  
     3. **FT-Transformer:** Deep Learning Transformer for Tabular Data  
       
  **Protocols:**  
     1. **Within-Dataset:** Train/Test on same dataset (80/20 split) to establish baseline  
     2. **Cross-Dataset:** Train on Dataset X  Test on Dataset Y to assess generalization  
     3. **Merged Datasets:** Train on X+Y  Test on Z to see if data diversity improves performance.

# Step 1: Set up

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Metrics & classical model
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score, classification_report, precision_score, recall_score
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.inspection import permutation_importance
from scipy import stats
import shap

# Deep Learning (PyTorch)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Weights & Biases
import wandb

In [ ]:
# Configuration
pd.set_option('display.max_columns', None)
RANDOM_SEED = 42
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Set this to True to enable logging
USE_WANDB = True
WANDB_PROJECT = 'phishing-feature-robustness'
print(f"Using device: {device}")
if USE_WANDB:
  wandb.login()

# step 2: define model

In [ ]:
from sklearn.preprocessing import StandardScaler

class SimpleFTTransformer(nn.Module):
  def __init__(self, num_numerical_features, embed_dim=32, num_heads=4, num_layers=3, dropout=0.1):
    super().__init__()
    # Feature Tokenizer: Treat numerical features as individual 'tokens' via linear projection
    self.feature_tokenizer = nn.ModuleList([
        nn.Linear(1, embed_dim) for _ in range(num_numerical_features)
    ])
    # CLS Token (learnable embedding for classification)
    self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
    # Transformer Encoder
    encoder_layer = nn.TransformerEncoderLayer(
        d_model=embed_dim,
        nhead=num_heads,
        dim_feedforward=embed_dim*2,
        dropout=dropout,
        batch_first=True
    )
    self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    # Classification Head
    self.mlp_head = nn.Sequential(
        nn.LayerNorm(embed_dim),
        nn.Linear(embed_dim, 1) # Binary output (logits)
    )

  def forward(self, x_numerical):
    batch_size = x_numerical.shape[0]

    # Tokenize: (Batch, Num_Features) -> (Batch, Num_Features, Embed_Dim)
    x_numerical = x_numerical.unsqueeze(-1)
    tokens = [layer(x_numerical[:, i]) for i, layer in enumerate(self.feature_tokenizer)]
    x = torch.stack(tokens, dim=1)

    # Prepend CLS token
    cls_tokens = self.cls_token.expand(batch_size, -1, -1)
    x = torch.cat((cls_tokens, x), dim=1)

    # Apply Transformer
    x = self.transformer(x)

    # Use CLS token output (index 0) for prediction
    cls_output = x[:, 0, :]
    return self.mlp_head(cls_output)

class TabularTransformerClassifier(BaseEstimator, ClassifierMixin):
    """Sklearn wrapper for the PyTorch FT-Transformer"""
    def __init__(self, epochs=20, batch_size=64, learning_rate=1e-3, use_wandb=False, patience=5, min_delta=0.001):
        self.epochs = epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.use_wandb = use_wandb
        self.patience = patience
        self.min_delta = min_delta
        self.model = None
        self.scaler = StandardScaler()
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def fit(self, X, y):
        # Scale features internally (important for neural nets)
        X_scaled = self.scaler.fit_transform(X)

        # Convert to Tensor
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(self.device)
        y_tensor = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1).to(self.device)

        # Initialize Model
        num_features = X.shape[1]
        self.model = SimpleFTTransformer(num_features).to(self.device)
        optimizer = optim.AdamW(self.model.parameters(), lr=self.learning_rate)
        criterion = nn.BCEWithLogitsLoss()

        # Training Loop
        self.model.train()
        dataset = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        print(f"Training FT-Transformer for {self.epochs} epochs...")
        best_loss = float('inf')
        patience_counter = 0

        for epoch in range(self.epochs):
            epoch_loss = 0
            for batch_X, batch_y in loader:
                optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()

            avg_loss = epoch_loss / len(loader)
            if self.use_wandb:
                wandb.log({'train_loss': avg_loss, 'epoch': epoch})

            # Early Stopping Check
            if avg_loss < best_loss - self.min_delta:
                best_loss = avg_loss
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= self.patience:
                    print(f"Early stopping triggered at epoch {epoch} (Loss: {avg_loss:.4f})")
                    break

        return self

    def predict_proba(self, X):
        self.model.eval()
        X_scaled = self.scaler.transform(X)
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            logits = self.model(X_tensor)
            probs = torch.sigmoid(logits).cpu().numpy()
        # Return (N, 2) array [Prob_0, Prob_1]
        return np.hstack([1 - probs, probs])

    def predict(self, X):
        probs = self.predict_proba(X)[:, 1]
        return (probs > 0.5).astype(int)

# Step 3: Load data

In [ ]:
DATA_DIR = './cleaned'

try:
    df_A = pd.read_csv(os.path.join(DATA_DIR, 'A_common.csv'))
    df_B = pd.read_csv(os.path.join(DATA_DIR, 'B_common.csv'))
    df_C = pd.read_csv(os.path.join(DATA_DIR, 'C_common.csv'))

    datasets = {
        'A': df_A,
        'B': df_B,
        'C': df_C,
    }
    print('Datasets loaded successfully.')
    print('Features:', list(df_A.columns))
except FileNotFoundError:
    print('Error: Dataset files not found. Please run the Preprocessing Notebook first.')

# Step 4: Define evaluation functions

In [ ]:
def evaluate_model(model, X_test, y_test, model_name, train_set_name, test_set_name, experiment_type):
        """Calculates Accuracy, F1, ROC-AUC, PR-AUC, Precision, and Recall."""
        y_pred = model.predict(X_test)

        # Handle probabilities for AUC
        if hasattr(model, 'predict_proba'):
            y_prob = model.predict_proba(X_test)[:, 1]
        else:
            y_prob = y_pred # Fallback if no proba

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc = roc_auc_score(y_test, y_prob)
        pr_auc = average_precision_score(y_test, y_prob)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)

        return {
            'Model': model_name,
            'Train_Set': train_set_name,
            'Test_Set': test_set_name,
            'Type': experiment_type,
            'Accuracy': acc,
            'F1_Score': f1,
            'ROC_AUC': roc,
            'PR_AUC': pr_auc,
            'Precision': precision,
            'Recall': recall,
        }

def get_X_y(df):
        X = df.drop(columns=['label'])
        y = df['label']
        return X, y

# Global Results Log
results_log = []

## Step 5. Protocol 1: Within-Dataset Baseline,
    Train on 80%, Test on 20% of the same dataset.

In [ ]:
MODELS_CONFIG = {
        'LogisticRegression': lambda: LogisticRegression(max_iter=2000, random_state=RANDOM_SEED),
        'RandomForest': lambda: RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED),
        'FT-Transformer': lambda: TabularTransformerClassifier(epochs=15, batch_size=64, learning_rate=1e-3, use_wandb=USE_WANDB),
    }

print("--- Protocol 1: Within-Dataset Baseline ---")

baseline_results = {}

for name, df in datasets.items():
        print(f"Processing Dataset {name}...")
        X, y = get_X_y(df)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

        for model_name, model_factory in MODELS_CONFIG.items():
            clf = model_factory()
            clf.fit(X_train, y_train)

            metrics = evaluate_model(clf, X_test, y_test, model_name, name, name, 'Baseline (Within)')
            results_log.append(metrics)

            if USE_WANDB:
                wandb.log(metrics)

            # Save baseline AUC for gap calculation later
            baseline_results[(model_name, name)] = metrics['ROC_AUC']

            print(f"   [{model_name}] AUC: {metrics['ROC_AUC']:.4f}")

## Step 6: Protocol 2: Cross-Dataset Generalization,
    Train on Dataset Source (100%), Test on Dataset Target (100%).

In [ ]:
print("--- Protocol 2: Cross-Dataset Generalization ---")

dataset_names = list(datasets.keys())

for source in dataset_names:
        for target in dataset_names:
            if source == target:
                continue

            print(f"Train {source} -> Test {target}")

            # Full datasets for transfer
            X_train, y_train = get_X_y(datasets[source])
            X_test, y_test = get_X_y(datasets[target])

            for model_name, model_factory in MODELS_CONFIG.items():
                clf = model_factory()
                clf.fit(X_train, y_train)

                metrics = evaluate_model(clf, X_test, y_test, model_name, source, target, 'Cross-Dataset')
                results_log.append(metrics)

                if USE_WANDB:
                    wandb.log(metrics)

                print(f"   [{model_name}] AUC: {metrics['ROC_AUC']:.4f}")

## Step 7: Protocol 3: Merged Datasets,
    Train on A+B, Test on C (and other permutations).

In [ ]:
print("--- Protocol 3: Merged Datasets ---")

combinations = [
        (['A', 'B'], 'C'),
        (['A', 'C'], 'B'),
        (['B', 'C'], 'A'),
    ]

for train_list, test_name in combinations:
        train_name = '+'.join(train_list)
        print(f"Train {train_name} -> Test {test_name}")

        # Merge Train Data
        df_train = pd.concat([datasets[d] for d in train_list], ignore_index=True)
        df_test = datasets[test_name]

        X_train, y_train = get_X_y(df_train)
        X_test, y_test = get_X_y(df_test)

        for model_name, model_factory in MODELS_CONFIG.items():
            clf = model_factory()
            clf.fit(X_train, y_train)

            metrics = evaluate_model(clf, X_test, y_test, model_name, train_name, test_name, 'Merged')
            results_log.append(metrics)

            if USE_WANDB:
                wandb.log(metrics)

            print(f"   [{model_name}] AUC: {metrics['ROC_AUC']:.4f}")

## Step 8: Analysis & Gap Calculation

> Calculate: $\\Delta AUC = AUC_{indataset} - AUC_{crossdataset}$


    

In [ ]:
results_df = pd.DataFrame(results_log)

def calculate_gap(row):
        # Only applicable for Cross-Dataset rows
        if row['Type'] == 'Cross-Dataset':
            # Retrieve the baseline AUC for this model on the *Source* dataset
            # (i.e., how well it performed when trained/tested on source)
            base_auc = baseline_results.get((row['Model'], row['Train_Set']), np.nan)
            return base_auc - row['ROC_AUC']
        return 0.0

results_df['Gen_Gap_AUC'] = results_df.apply(calculate_gap, axis=1)

# Display Top Results
print("Sample Results:")
display(results_df.head(10))

# Save Results
output_path = os.path.join(DATA_DIR, 'experiment_results_log.csv')
results_df.to_csv(output_path, index=False)
print(f"Full results saved to {output_path}")

## Step 9: :Visualization: Model Performance Comparison,
    Comparing ROC-AUC across different models and protocols.

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=results_df, x='Train_Set', y='ROC_AUC', hue='Model')
plt.title('Model Performance (ROC-AUC) Across Training Sets')
plt.ylabel('ROC-AUC')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Step 10. Feature Importance & Robustness Analysis
  This section computes:  
    1. **Permutation Importance & SHAP:** For each model trained in Protocol 1  
    2. **Robustness Score (R):**  
    3. **Spearman Rank Correlation:** Between feature importance and SHAP stability

> R = Mean / (1 + Std)  

    

In [ ]:
def calculate_robustness(imp_df):
    """Calculates R = Mean / (1 + Std) across datasets for each feature."""
    mean_imp = imp_df.mean(axis=1)
    std_imp = imp_df.std(axis=1)
    robustness_score = mean_imp / (1 + std_imp)
    return mean_imp, std_imp, robustness_score

print("--- Starting Feature Importance Analysis ---")

feature_names = list(datasets['A'].drop(columns=['label']).columns)

# Storage for importance matrices
all_model_importances = {}

for model_name in MODELS_CONFIG.keys():
    print(f"\nAnalyzing {model_name}...")

    # DataFrame to store importance of each feature for Datasets A, B, C
    model_imp_df = pd.DataFrame(index=feature_names)

    for dataset_name in datasets.keys():
        # Instantiate and fit model since trained_baselines wasn't saved
        clf = MODELS_CONFIG[model_name]()
        X_data, y_data = get_X_y(datasets[dataset_name])
        clf.fit(X_data, y_data)

        # Use a small subset for SHAP/Permutation speed
        X_sample = X_data.sample(min(500, len(X_data)), random_state=RANDOM_SEED)
        y_sample = y_data.loc[X_sample.index]

        # 1. Permutation Importance (Model Agnostic & Robust)
        perm_results = permutation_importance(clf, X_sample, y_sample, n_repeats=5, random_state=RANDOM_SEED)
        importances = perm_results.importances_mean

        # 2. SHAP (Model Specific)
        try:
            if model_name == 'RandomForest':
                explainer = shap.TreeExplainer(clf)
                shap_values = explainer.shap_values(X_sample)
                if isinstance(shap_values, list):
                    shap_vals = np.abs(shap_values[1]).mean(axis=0)
                else:
                    shap_vals = np.abs(shap_values).mean(axis=0)

            elif model_name == 'LogisticRegression':
                explainer = shap.LinearExplainer(clf, X_sample)
                shap_values = explainer.shap_values(X_sample)
                shap_vals = np.abs(shap_values).mean(axis=0)

        except Exception as e:
            pass

        model_imp_df[dataset_name] = importances

    # Calculate Robustness Score R
    mean_imp, std_imp, r_score = calculate_robustness(model_imp_df)

    model_imp_df['Mean_Imp'] = mean_imp
    model_imp_df['Std_Imp'] = std_imp
    model_imp_df['Robustness_R'] = r_score

    spearman_corr, _ = stats.spearmanr(model_imp_df['Mean_Imp'], model_imp_df['Std_Imp'])
    print(f"   Spearman Corr (Imp vs Std): {spearman_corr:.4f}")

    all_model_importances[model_name] = model_imp_df

    print(f"   Top 5 Robust Features (High R):")
    display(model_imp_df.sort_values(by='Robustness_R', ascending=False).head(5))

### Visualization: SHAP values
    "The plot below shows which features are most important and how they impact model (logistic & RandomForest) prediction "

In [ ]:
# Configuration for Visualization
viz_model_name = 'RandomForest' # You can change this to 'LogisticRegression'
viz_dataset_name = 'A'          # You can change this to 'B' or 'C'

print(f"Generating SHAP visualization for {viz_model_name} on Dataset {viz_dataset_name}...")

# 1. Get Data and Train Model
# We retrain here to ensure we have the fitted model object available
X, y = get_X_y(datasets[viz_dataset_name])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

clf = MODELS_CONFIG[viz_model_name]()
clf.fit(X_train, y_train)

# 2. Compute SHAP Values
# Use a sample of the test set for faster computation
X_sample = X_test.sample(min(200, len(X_test)), random_state=RANDOM_SEED)

if viz_model_name == 'RandomForest':
    explainer = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(X_sample)
    # TreeExplainer for binary classification often returns a list [class0, class1]
    # We select index 1 for the positive class (Phishing)
    if isinstance(shap_values, list):
        shap_vals_plot = shap_values[1]
    else:
        shap_vals_plot = shap_values

elif viz_model_name == 'LogisticRegression':
    # LinearExplainer needs a background dataset (X_train summary)
    # We summarize X_train to keep it fast
    explainer = shap.LinearExplainer(clf, X_train)
    shap_values = explainer.shap_values(X_sample)
    shap_vals_plot = shap_values

# 3. Create Summary Plot
plt.figure(figsize=(10, 6))
plt.title(f"SHAP Summary Plot: {viz_model_name} - Dataset {viz_dataset_name}")
shap.summary_plot(shap_vals_plot, X_sample, show=True)

### Visualization: Robustness vs Importance
    "The plot below shows Feature Importance (X) vs. Robustness Score (Y). Features in the top-right are both **highly important** and **highly robust across datasets**."

In [ ]:
for model_name, df_imp in all_model_importances.items():
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df_imp, x='Mean_Imp', y='Robustness_R', s=100)

    # Annotate top features
    top_features = df_imp.sort_values('Robustness_R', ascending=False).head(5)
    for idx, row in top_features.iterrows():
        plt.text(row['Mean_Imp'], row['Robustness_R'], idx, fontsize=9, ha='left')

    plt.title(f'Feature Robustness Analysis: {model_name}')
    plt.xlabel('Mean Feature Importance (across datasets)')
    plt.ylabel('Robustness Score R (Mean / 1+Std)')
    plt.grid(True, alpha=0.3)
    plt.show()